<a href="https://colab.research.google.com/github/trendleader/AI/blob/main/LangChain_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 18.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.80
    Uninstalling langchain-core-0.3.80:
      Successfully uninstalled langchain-core-0.3.80
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.27
    Uninstalling langchain-0.3.27:
      Successfully uninstalled langchain-0.3.27


In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
# simple agent
from langchain.agents import create_agent

def get_weather(city:str) -> str:
  """Get weather for a given city"""
  return f"Its always sunny in {city}"

agent = create_agent(
    model="gpt-5-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant who answers the question and doesnot ask any followup questions"
)

res = agent.invoke({"messages":[{"role":"user","content":"What is the weather in Tokyo?"}]})
print(res["messages"][-1].content)

According to the weather service, it's always sunny in Tokyo.


In [ ]:
# structured output
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel

class Weather(BaseModel):
  temperature:float
  condition:str

def get_weather(city:str) -> str:
  """Get the weather for a city."""
  return f"it's sunny and 70 degrees in {city}"


agent = create_agent(
    model="gpt-5-mini",
    tools=[get_weather],
    response_format=ToolStrategy(Weather)
    )

res = agent.invoke({"messages":[{"role":"user","content":"What is the weather in Tokyo?"}]})
print(res["messages"][-1].content)

Returning structured response: temperature=70.0 condition='sunny'


In [ ]:
# Real world agent
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""



In [ ]:
# Create tools
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class Context:
  user_id:str

@tool
def get_weather(city:str) -> str:
  """Get weather for a given city"""
  return f"Its always sunny in {city}"

@tool
def get_user_location(runtime:ToolRuntime[Context]) -> str:
  """Get user location based on user id"""
  user_id = runtime.context.user_id
  return "Florida" if user_id=="1" else "San Francisco"


In [ ]:
# initiate chat model
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="gpt-5-mini",
    temperature=0.5,
    timeout=10,
    max_tokens=1000
)


In [ ]:
# Define response format
from dataclasses import dataclass

@dataclass
class ResponseFormat:
  """Response schema for the agent"""
  punny_response:str
  weather_conditions:str|None = None

In [ ]:
# Add Memory

from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [ ]:
# Create and run the agent
agent = create_agent(
    model = model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location,get_weather],
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

# thread is a unique identifier for a conversation
config = {"configurable":{"thread_id":1}}

response = agent.invoke({
    "messages":[{"role":"user","content":"What is the weather outside? Find my location and then tell me about the weather"}]},
    config=config,
    context=Context(user_id="1")
)

print(response["structured_response"])

response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])


ResponseFormat(punny_response="I found you — you're in Florida, so the forecast is sun-sational! No clouds about it: it’s always sunny in Florida. Don’t foget your shades — that’s a bright idea!", weather_conditions="It's always sunny in Florida.")
ResponseFormat(punny_response="You're welcome! Glad I could brighten your day — I'm always here to “weather” your questions. Want an hourly or 7-day outlook?", weather_conditions=None)
